# LangChain + CoordiNode: Graph Chain

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/structured-world/coordinode-python/blob/main/demo/notebooks/02_langchain_graph_chain.ipynb)

Demonstrates `CoordinodeGraph` as a Knowledge Graph backend for LangChain.

**What works right now:**
- `graph.query()` — arbitrary Cypher pass-through
- `graph.schema` / `refresh_schema()` — live graph schema
- `add_graph_documents()` — add Nodes + Relationships from a LangChain `GraphDocument`
- `GraphCypherQAChain` — LLM generates Cypher from a natural-language question *(requires `OPENAI_API_KEY`)*

**Environments:**
- **Google Colab** — uses `coordinode-embedded` (in-process Rust engine, no server needed). First run compiles from source (~5 min); subsequent runs use the pip cache.
- **Local / Docker Compose** — connects to a running CoordiNode server via gRPC.

## Install dependencies

In [ ]:
import sys, subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Install Rust toolchain — required to build coordinode-embedded from source
    subprocess.run('curl --proto "=https" --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y -q', shell=True, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "maturin"], check=True)
    # Install coordinode-embedded from GitHub source (~5 min first run, cached after)
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "git+https://github.com/structured-world/coordinode-python.git#subdirectory=coordinode-embedded",
            "langchain-coordinode",
            "langchain",
            "langchain-openai",
            "langchain-community",
            "nest_asyncio",
        ],
        check=True,
    )
else:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "coordinode",
            "langchain-coordinode",
            "langchain",
            "langchain-openai",
            "langchain-community",
            "nest_asyncio",
        ],
        check=True,
    )

import nest_asyncio

nest_asyncio.apply()

print("SDK installed")

## Adapter for embedded mode

In Colab we use `LocalClient` (embedded engine) which has the same `.cypher()` API as
`CoordinodeClient`. The `_EmbeddedAdapter` below adds the extra methods that
`CoordinodeGraph` expects when it receives a pre-built `client=` object.

In [ ]:
class _EmbeddedAdapter:
    """Thin wrapper around LocalClient that adds CoordinodeClient-compatible methods."""

    def __init__(self, local_client):
        self._lc = local_client

    def cypher(self, query, params=None):
        return self._lc.cypher(query, params or {})

    def get_schema_text(self):
        lbls = self._lc.cypher("MATCH (n) UNWIND labels(n) AS lbl RETURN DISTINCT lbl ORDER BY lbl")
        rels = self._lc.cypher("MATCH ()-[r]->() RETURN DISTINCT type(r) AS t ORDER BY t")
        lines = ["Node labels:"]
        for r in lbls:
            lines.append(f"  - {r['lbl']}")
        lines.append("\nEdge types:")
        for r in rels:
            lines.append(f"  - {r['t']}")
        return "\n".join(lines)

    def vector_search(self, **kwargs):
        # Not implemented in embedded mode — vector index requires a running CoordiNode server.

        return []

    def close(self):
        self._lc.close()

    def get_labels(self):
        # Schema introspection not available in embedded mode.

        return []

    def get_edge_types(self):
        # Schema introspection not available in embedded mode.
        return []

## Connect to CoordiNode

In [ ]:
import os, socket, subprocess, time

if IN_COLAB:
    from coordinode_embedded import LocalClient

    _lc = LocalClient(":memory:")
    client = _EmbeddedAdapter(_lc)
    print("Using embedded LocalClient (in-process)")
else:
    GRPC_PORT = int(os.environ.get("COORDINODE_PORT", "7080"))
    IMAGE = "ghcr.io/structured-world/coordinode:latest"

    def _port_open(port):
        try:
            with socket.create_connection(("127.0.0.1", port), timeout=1):
                return True
        except OSError:
            return False

    if os.environ.get("COORDINODE_ADDR"):
        COORDINODE_ADDR = os.environ["COORDINODE_ADDR"]
        print(f"Using COORDINODE_ADDR from environment: {COORDINODE_ADDR}")
    elif _port_open(GRPC_PORT):
        print(f"CoordiNode already reachable on :{GRPC_PORT}")
        COORDINODE_ADDR = f"localhost:{GRPC_PORT}"
    else:
        print(f"Starting CoordiNode via Docker on :{GRPC_PORT} …")
        proc = subprocess.run(
            ["docker", "run", "-d", "--rm", "-p", f"{GRPC_PORT}:7080", IMAGE], capture_output=True, text=True
        )
        if proc.returncode != 0:
            raise RuntimeError("docker run failed: " + proc.stderr)
        container_id = proc.stdout.strip()
        for _ in range(30):
            if _port_open(GRPC_PORT):
                break
            time.sleep(1)
        else:
            subprocess.run(["docker", "stop", container_id])
            raise RuntimeError("CoordiNode did not start in 30 s")
        COORDINODE_ADDR = f"localhost:{GRPC_PORT}"
        print(f"CoordiNode ready at {COORDINODE_ADDR}")

    from coordinode import CoordinodeClient

    client = CoordinodeClient(COORDINODE_ADDR)
    print(f"Connected to {COORDINODE_ADDR}")

## Create the graph store

Pass the pre-built `client=` object so the store doesn't open a second connection.

In [ ]:
import os, uuid
from langchain_coordinode import CoordinodeGraph
from langchain_community.graphs.graph_document import GraphDocument, Node, Relationship
from langchain_core.documents import Document

graph = CoordinodeGraph(client=client)
print("Connected. Schema preview:")
print(graph.schema[:300])

## 1. add_graph_documents

LangChain `GraphDocument` wraps nodes and relationships from an LLM extraction pipeline.
Here we build one manually to show the insert path.

In [ ]:
tag = uuid.uuid4().hex[:6]

nodes = [
    Node(id=f"Turing-{tag}", type="Scientist", properties={"born": 1912, "field": "computer science"}),
    Node(id=f"Shannon-{tag}", type="Scientist", properties={"born": 1916, "field": "information theory"}),
    Node(id=f"Cryptography-{tag}", type="Field", properties={"era": "modern"}),
]
rels = [
    Relationship(source=nodes[0], target=nodes[2], type="FOUNDED", properties={"year": 1936}),
    Relationship(source=nodes[1], target=nodes[2], type="CONTRIBUTED_TO"),
    Relationship(source=nodes[0], target=nodes[1], type="INFLUENCED"),
]
doc = GraphDocument(nodes=nodes, relationships=rels, source=Document(page_content="Turing and Shannon"))
graph.add_graph_documents([doc])
print("Documents added")

## 2. query — direct Cypher

In [ ]:
rows = graph.query(
    "MATCH (s:Scientist)-[r]->(f:Field)"
    " WHERE s.name STARTS WITH $prefix"
    " RETURN s.name AS scientist, type(r) AS relation, f.name AS field",
    params={"prefix": f"Turing-{tag[:4]}"},
)
print("Scientists → Fields:")
for r in rows:
    print(f"  {r['scientist']}  --[{r['relation']}]-->  {r['field']}")

## 3. refresh_schema — structured_schema dict

In [ ]:
graph.refresh_schema()
print("node_props keys:", list(graph.structured_schema.get("node_props", {}).keys())[:10])
print("relationships:", graph.structured_schema.get("relationships", [])[:5])

## 4. Idempotency check

`add_graph_documents` uses MERGE internally — adding the same document twice must not
create duplicate edges.

In [ ]:
graph.add_graph_documents([doc])  # second upsert — must not create a duplicate edge
cnt = graph.query(
    "MATCH (a {name: $src})-[r:FOUNDED]->(b {name: $dst}) RETURN count(r) AS cnt",
    params={"src": f"Turing-{tag}", "dst": f"Cryptography-{tag}"},
)
print("FOUNDED edge count after double upsert (expect 1):", cnt[0]["cnt"])

## 5. GraphCypherQAChain — LLM-powered Cypher (optional)

> **This section requires `OPENAI_API_KEY`.** Set it in your environment or via
> `os.environ['OPENAI_API_KEY'] = 'sk-...'` before running.
> The cell is skipped automatically when the key is absent.

In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    print(
        'Skipping: OPENAI_API_KEY is not set. Set it via os.environ["OPENAI_API_KEY"] = "sk-..." and re-run this cell.'
    )
else:
    from langchain.chains import GraphCypherQAChain
    from langchain_openai import ChatOpenAI

    chain = GraphCypherQAChain.from_llm(
        ChatOpenAI(model="gpt-4o-mini", temperature=0),
        graph=graph,
        verbose=True,
    )
    result = chain.invoke("Who influenced Shannon?")
    print("Answer:", result["result"])

## Cleanup

In [ ]:
graph.query("MATCH (n) WHERE n.name ENDS WITH $tag DETACH DELETE n", params={"tag": tag})
print("Cleaned up")
graph.close()